# **# BIBLIOTECAS E MÓDULOS**

In [1]:

import heapq
import numpy as np
import pandas as pd
import openpyxl as op
from openpyxl import load_workbook

ARQ = "0_7_7_DADOS_IBOV_2010_2019_dt.xlsx"

N_ATIVOS = 1

# **# RETORNOS MENSAIS E ANUAIS - IBOVESPA**

In [2]:

def carregar_base_ibov(caminho, sheet=None):
    """
    Lê a planilha e retorna uma Série de preços do IBOV (índice datetime).
    Tenta identificar colunas 'Date' e a coluna de preço (prioridade: '^BVSP', 'IBOV', 'Close', etc.).
    Trata datas como seriais Excel ou strings e limpa linhas não numéricas.
    """
    df = pd.read_excel(caminho, sheet_name=(0 if sheet is None else sheet), engine="openpyxl")
    df.columns = [str(c).strip() for c in df.columns]

    # --- identifica coluna de data ---
    col_date = None
    for c in df.columns:
        if c.lower() in ("date", "data", "dt"):
            col_date = c
            break
    if col_date is None:
        raise ValueError("Coluna de data ('Date' / 'Data' / 'dt') não encontrada no Excel.")

    # --- identifica coluna de preço ---
    preferidas = ["^BVSP", "IBOV", "Close", "Adj Close", "Preço", "Price", "Fechamento"]
    col_price = None
    for c in preferidas:
        if c in df.columns:
            col_price = c
            break
    if col_price is None:
        # fallback: primeira coluna que não é a de data
        candidatos = [c for c in df.columns if c != col_date]
        if not candidatos:
            raise ValueError("Não encontrei coluna de preço no arquivo.")
        col_price = candidatos[0]

    # --- converte datas ---
    if np.issubdtype(df[col_date].dtype, np.number):
        # números seriais do Excel (origem 1899-12-30)
        df["Date"] = pd.to_datetime(df[col_date], unit="d", origin="1899-12-30", errors="coerce")
    else:
        df["Date"] = pd.to_datetime(df[col_date], errors="coerce")

    # --- força preço numérico e remove linhas ruins ---
    df["Price"] = pd.to_numeric(df[col_price], errors="coerce")

    # remove linhas sem data ou sem preço, duplicadas de data; ordena
    df = df.dropna(subset=["Date", "Price"]).drop_duplicates(subset=["Date"]).sort_values("Date")

    # constrói série
    s = df.set_index("Date")["Price"].astype(float).sort_index()
    return s

# ----------------------- CÁLCULOS -----------------------

def retornos_mensais(series):
    """
    Retornos mensais via último dia útil do mês (ME) sobre o mês anterior.
    Retorna DataFrame com colunas: 'Ano', 'Mes', 'Retorno Mensal'.
    """
    v_m = series.resample("ME").last().dropna()
    ret_m = v_m.pct_change().dropna()
    ret_m_df = (
        ret_m.to_frame("Retorno Mensal")
            .assign(Ano=lambda d: d.index.year, Mes=lambda d: d.index.month)
            .loc[:, ["Ano", "Mes", "Retorno Mensal"]]
            .reset_index(drop=True)
    )
    return ret_m_df

def retornos_anuais_jan_dez(series):
    """
    Retorno anual (dentro do ano): do 1º pregão de JAN ao último pregão de DEZ.
    Retorna DataFrame com colunas: 'Ano', 'Retorno Anual (Jan→Dez)'.
    """
    # agrupa por ano e pega primeiro e último preço ÚTEIS do ano
    first_by_year = series.groupby(series.index.year).first()
    last_by_year  = series.groupby(series.index.year).last()
    # retorno dentro do ano
    ret_y = (last_by_year / first_by_year - 1.0).dropna()
    ret_y_df = ret_y.to_frame("Retorno Anual (Jan→Dez)").rename_axis("Ano").reset_index()
    return ret_y_df

# ----------------------- EXECUÇÃO -----------------------

if __name__ == "__main__":
    pd.set_option("display.max_rows", 200)
    pd.set_option("display.width", 160)

    # 1) Carrega série do IBOV
    ibov = carregar_base_ibov(ARQ)

    # 2) Calcula retornos
    ret_mensal_df = retornos_mensais(ibov)
    ret_anual_df  = retornos_anuais_jan_dez(ibov)

    # 3) Impressões
    print("\n=== IBOV (^BVSP) — Retornos ===")
    print(f"Período na base: {ibov.index.min().date()} até {ibov.index.max().date()}  |  N° pregões: {len(ibov)}")

    print("\n--- Retornos ANUAIS (Jan→Dez) ---")
    # versão formatada para leitura (2 casas em %)
    _fmt_anual = ret_anual_df.copy()
    _fmt_anual["Retorno Anual (Jan→Dez)"] = (_fmt_anual["Retorno Anual (Jan→Dez)"] * 100).map(lambda x: f"{x:.2f}%")
    print(_fmt_anual.to_string(index=False))

    print("\n--- Retornos MENSAIS ---")
    _fmt_mensal = ret_mensal_df.copy()
    _fmt_mensal["Retorno Mensal"] = (_fmt_mensal["Retorno Mensal"] * 100).map(lambda x: f"{x:.2f}%")
    print(_fmt_mensal.to_string(index=False))



=== IBOV (^BVSP) — Retornos ===
Período na base: 2010-01-04 até 2019-12-30  |  N° pregões: 2471

--- Retornos ANUAIS (Jan→Dez) ---
 Ano Retorno Anual (Jan→Dez)
2010                  -1.06%
2011                 -18.88%
2012                   2.85%
2013                 -17.65%
2014                  -0.66%
2015                 -10.64%
2016                  42.92%
2017                  28.21%
2018                  12.83%
2019                  27.42%

--- Retornos MENSAIS ---
 Ano  Mes Retorno Mensal
2010    2          1.68%
2010    3          5.82%
2010    4         -4.04%
2010    5         -6.64%
2010    6         -3.35%
2010    7         10.80%
2010    8         -3.51%
2010    9          6.58%
2010   10          1.79%
2010   11         -4.20%
2010   12          2.36%
2011    1         -3.94%
2011    2          1.21%
2011    3          1.79%
2011    4         -3.58%
2011    5         -2.29%
2011    6         -3.43%
2011    7         -5.74%
2011    8         -3.96%
2011    9         -7.38

# **# VOLATILIDADE ANUALIZADA EM PORCENTAGEM - IBOVESPA**

In [3]:
def carregar_base_ibov(caminho, sheet=None):
    """
    Lê a planilha e retorna uma Série de preços do IBOV (^BVSP) com índice datetime.
    Identifica colunas de 'Date' e preço (prioriza '^BVSP'); trata datas seriais do Excel.
    """
    df = pd.read_excel(caminho, sheet_name=(0 if sheet is None else sheet), engine="openpyxl")
    df.columns = [str(c).strip() for c in df.columns]

    # Identifica coluna de data
    col_date = None
    for c in df.columns:
        if c.lower() in ("date", "data", "dt"):
            col_date = c
            break
    if col_date is None:
        raise ValueError("Coluna de data ('Date'/'Data'/'dt') não encontrada.")

    # Identifica coluna de preço (preferências)
    preferidas = ["^BVSP", "IBOV", "Close", "Adj Close", "Preço", "Price", "Fechamento"]
    col_price = next((c for c in preferidas if c in df.columns), None)
    if col_price is None:
        # fallback: 1ª coluna que não é data
        cand = [c for c in df.columns if c != col_date]
        if not cand:
            raise ValueError("Coluna de preço não encontrada.")
        col_price = cand[0]

    # Converte datas (suporta serial Excel e string)
    if np.issubdtype(df[col_date].dtype, np.number):
        df["Date"] = pd.to_datetime(df[col_date], unit="d", origin="1899-12-30", errors="coerce")
    else:
        df["Date"] = pd.to_datetime(df[col_date], errors="coerce")

    # Preço numérico; remove linhas ruins; ordena por data
    df["Price"] = pd.to_numeric(df[col_price], errors="coerce")
    df = df.dropna(subset=["Date", "Price"]).drop_duplicates(subset=["Date"]).sort_values("Date")

    # Série final
    s = df.set_index("Date")["Price"].astype(float).sort_index()
    return s

# ----------------------- VOLATILIDADES -----------------------

def vol_anualizada_diaria(ret_diario: pd.Series) -> float:
    """Vol anualizada por base diária: std × sqrt(252)."""
    if len(ret_diario) < 2:
        return np.nan
    return float(ret_diario.std(ddof=1) * np.sqrt(252))

def vol_anualizada_mensal(ret_mensal: pd.Series) -> float:
    """Vol anualizada por base mensal: std × sqrt(12)."""
    if len(ret_mensal) < 2:
        return np.nan
    return float(ret_mensal.std(ddof=1) * np.sqrt(12))

def calcular_volatilidades(serie_preco: pd.Series):
    """
    Retorna:
      - vol_total_d, vol_total_m: volatilidades do período completo (base diária/mensal)
      - vol_anual_df: DataFrame com vol anualizada por ano (diária e mensal)
    """
    # Retornos diários
    ret_d = serie_preco.pct_change().dropna()

    # Retornos mensais pelo último dia útil do mês
    valor_m = serie_preco.resample("ME").last().dropna()
    ret_m = valor_m.pct_change().dropna()

    # Período completo
    vol_total_d = vol_anualizada_diaria(ret_d)
    vol_total_m = vol_anualizada_mensal(ret_m)

    # Por ano (base diária)
    vol_ano_d = []
    for ano, sub in ret_d.groupby(ret_d.index.year):
        if len(sub) >= 2:
            vol_ano_d.append({"Ano": int(ano), "Vol Anualizada (base diária)": vol_anualizada_diaria(sub)})
    vol_ano_d = pd.DataFrame(vol_ano_d).sort_values("Ano")

    # Por ano (base mensal)
    vol_ano_m = []
    for ano, sub in ret_m.groupby(ret_m.index.year):
        if len(sub) >= 2:
            vol_ano_m.append({"Ano": int(ano), "Vol Anualizada (base mensal)": vol_anualizada_mensal(sub)})
    vol_ano_m = pd.DataFrame(vol_ano_m).sort_values("Ano")

    # Consolida
    vol_anual_df = pd.merge(vol_ano_d, vol_ano_m, on="Ano", how="outer").sort_values("Ano").reset_index(drop=True)
    return vol_total_d, vol_total_m, vol_anual_df

# ----------------------- EXECUÇÃO -----------------------

if __name__ == "__main__":
    pd.set_option("display.max_rows", 200)
    pd.set_option("display.width", 160)

    # 1) Carrega série do IBOV (^BVSP)
    ibov = carregar_base_ibov(ARQ)

    # 2) Calcula volatilidades
    vol_total_d, vol_total_m, vol_anual = calcular_volatilidades(ibov)

    # 3) Formatação para impressão (percentual com 2 casas)
    vol_anual_fmt = vol_anual.copy()
    for c in ["Vol Anualizada (base diária)", "Vol Anualizada (base mensal)"]:
        if c in vol_anual_fmt.columns:
            vol_anual_fmt[c] = (vol_anual_fmt[c] * 100).map(lambda x: f"{x:.2f}%")

    print("\n=== IBOV (^BVSP) — Volatilidade Anualizada ===")
    print(f"Período na base: {ibov.index.min().date()} até {ibov.index.max().date()}  |  Nº pregões: {len(ibov)}")

    print("\n--- Por ano (anualizada) ---")
    if not vol_anual_fmt.empty:
        print(vol_anual_fmt.to_string(index=False))
    else:
        print("[Aviso] Não foi possível calcular a volatilidade anualizada por ano (dados insuficientes).")

    print("\n--- Período completo ---")
    print(f"Vol anualizada (base diária):  {vol_total_d*100:.2f}%")
    print(f"Vol anualizada (base mensal):  {vol_total_m*100:.2f}%")


=== IBOV (^BVSP) — Volatilidade Anualizada ===
Período na base: 2010-01-04 até 2019-12-30  |  Nº pregões: 2471

--- Por ano (anualizada) ---
 Ano Vol Anualizada (base diária) Vol Anualizada (base mensal)
2010                       20.30%                       19.04%
2011                       24.68%                       16.93%
2012                       21.89%                       20.22%
2013                       20.45%                       15.30%
2014                       25.22%                       22.38%
2015                       23.30%                       20.42%
2016                       26.65%                       28.39%
2017                       19.03%                       14.25%
2018                       22.21%                       22.45%
2019                       18.00%                       12.41%

--- Período completo ---
Vol anualizada (base diária):  22.33%
Vol anualizada (base mensal):  19.67%


# **# DRAWDOWN MÁXIMO - IBOVESPA**

Abaixo está um complemento direto ao seu script de volatilidade para calcular e imprimir o Max Drawdown (MDD):

MDD do período completo (valor e datas de peak → trough).
MDD por ano-calendário.
Tudo em porcentagem com 2 casas decimais.


Definição: Drawdown em ttt = Vtmax⁡s≤tVs−1\frac{V_t}{\max_{s \le t} V_s} - 1maxs≤t​Vs​Vt​​−1.
O Max Drawdown é o menor (mais negativo) valor dessa série.



In [4]:

def carregar_base_ibov(caminho, sheet=None):
    """
    Lê a planilha e retorna uma Série de preços do IBOV (^BVSP) com índice datetime.
    Identifica colunas de 'Date' e preço (prioriza '^BVSP'); trata datas seriais do Excel.
    """
    df = pd.read_excel(caminho, sheet_name=(0 if sheet is None else sheet), engine="openpyxl")
    df.columns = [str(c).strip() for c in df.columns]

    # Identifica coluna de data
    col_date = None
    for c in df.columns:
        if c.lower() in ("date", "data", "dt"):
            col_date = c
            break
    if col_date is None:
        raise ValueError("Coluna de data ('Date'/'Data'/'dt') não encontrada.")

    # Identifica coluna de preço (preferências)
    preferidas = ["^BVSP", "IBOV", "Close", "Adj Close", "Preço", "Price", "Fechamento"]
    col_price = next((c for c in preferidas if c in df.columns), None)
    if col_price is None:
        cand = [c for c in df.columns if c != col_date]
        if not cand:
            raise ValueError("Coluna de preço não encontrada.")
        col_price = cand[0]

    # Converte datas (suporta serial Excel e string)
    if np.issubdtype(df[col_date].dtype, np.number):
        df["Date"] = pd.to_datetime(df[col_date], unit="d", origin="1899-12-30", errors="coerce")
    else:
        df["Date"] = pd.to_datetime(df[col_date], errors="coerce")

    # Preço numérico; remove linhas ruins; ordena por data
    df["Price"] = pd.to_numeric(df[col_price], errors="coerce")
    df = df.dropna(subset=["Date", "Price"]).drop_duplicates(subset=["Date"]).sort_values("Date")

    # Série final
    s = df.set_index("Date")["Price"].astype(float).sort_index()
    return s

# ----------------------- CÁLCULO DO MDD -----------------------

def max_drawdown(serie_val: pd.Series):
    """
    Retorna:
      - mdd: float (ex.: -0.325 corresponde a -32,50%)
      - data_peak: data do pico que antecede o vale do MDD
      - data_trough: data do vale (onde o drawdown é máximo)
      - dd_series: série de drawdown ao longo do tempo
    """
    v = serie_val.astype(float).copy()
    roll_max = v.cummax()
    dd = (v / roll_max) - 1.0

    # MDD (valor mínimo)
    mdd = float(dd.min())

    # Datas de peak -> trough
    trough_idx = dd.idxmin()
    peak_idx = v.loc[:trough_idx].idxmax()

    return mdd, peak_idx, trough_idx, dd

# ----------------------- EXECUÇÃO -----------------------

if __name__ == "__main__":
    pd.set_option("display.max_rows", 200)
    pd.set_option("display.width", 160)

    # 1) Carrega série do IBOV (^BVSP)
    ibov = carregar_base_ibov(ARQ)

    # 2) MDD do período completo
    mdd_total, peak_total, trough_total, dd_series = max_drawdown(ibov)

    # 3) MDD por ano
    mdd_por_ano = []
    for ano, sub in ibov.groupby(ibov.index.year):
        if len(sub) >= 2:
            mdd_a, peak_a, trough_a, _ = max_drawdown(sub)
            mdd_por_ano.append({
                "Ano": int(ano),
                "MDD": mdd_a,
                "Peak": pd.to_datetime(peak_a).date(),
                "Trough": pd.to_datetime(trough_a).date()
            })
    mdd_por_ano = pd.DataFrame(mdd_por_ano).sort_values("Ano", ascending=True)

    # 4) Impressão
    print("\n=== IBOV (^BVSP) — MAX DRAWDOWN (MDD) ===")
    print(f"Período na base: {ibov.index.min().date()} até {ibov.index.max().date()}  |  Nº pregões: {len(ibov)}")

    # Período completo
    print(f"\nMDD do período: {mdd_total*100:.2f}%")
    print(f"  Pico (peak)  : {pd.to_datetime(peak_total).date()}")
    print(f"  Vale (trough): {pd.to_datetime(trough_total).date()}")

    # Por ano (em % com 2 casas)
    if not mdd_por_ano.empty:
        mdd_por_ano_fmt = mdd_por_ano.copy()
        mdd_por_ano_fmt["MDD"] = (mdd_por_ano_fmt["MDD"] * 100).map(lambda x: f"{x:.2f}%")
        print("\n--- MDD por ano ---")
        print(mdd_por_ano_fmt.to_string(index=False))


=== IBOV (^BVSP) — MAX DRAWDOWN (MDD) ===
Período na base: 2010-01-04 até 2019-12-30  |  Nº pregões: 2471

MDD do período: -48.63%
  Pico (peak)  : 2010-11-04
  Vale (trough): 2016-01-26

--- MDD por ano ---
 Ano     MDD       Peak     Trough
2010 -18.94% 2010-04-08 2010-05-20
2011 -32.06% 2011-01-12 2011-08-08
2012 -23.27% 2012-03-13 2012-06-05
2013 -28.85% 2013-01-03 2013-07-03
2014 -24.05% 2014-09-02 2014-12-16
2015 -25.58% 2015-05-05 2015-12-21
2016 -12.04% 2016-10-31 2016-12-19
2017 -12.01% 2017-02-21 2017-06-21
2018 -20.35% 2018-02-26 2018-06-18
2019 -10.00% 2019-03-18 2019-05-17


# **# ÍNDICE SHARPE - IBOVESPA**

Abaixo está um complemento direto para o script que você já tem (o de volatilidade e MDD). Este trecho calcula o Índice de Sharpe:

Sharpe anualizado (base diária) usando retornos diários do portfólio e um rate livre de risco anual (configurável; por padrão 0,00%).
Sharpe anualizado (base mensal) usando retornos mensais.
Sharpe por ano-calendário e para o período completo.


Fórmula (anualizado):
Sharpe=μ−rfσ\text{Sharpe} = \frac{\mu - r_f}{\sigma}Sharpe=σμ−rf​​
com μ\muμ e σ\sigmaσ sendo a média e o desvio-padrão dos excess returns (retorno da carteira menos o retorno livre de risco), e anualização via 252\sqrt{252}252​ (diário) ou 12\sqrt{12}12​ (mensal).

In [5]:
# =========================================
# IBOVESPA (2010–2019) – Índice de Sharpe (anualizado)
# - Arquivo de entrada: 5_1_DADOS_IBOV_2010_2019_dt.xlsx
# - Limpeza: datas (seriais do Excel ou strings), coluna de preço (^BVSP)
# - Saída: Sharpe anualizado por ano e do período (base diária e mensal)
# =========================================

import numpy as np
import pandas as pd

ARQ = "5_1_DADOS_IBOV_2010_2019_dt.xlsx"  # coloque o arquivo na mesma pasta

# ----------- Configuração da taxa livre de risco -----------
# Taxa livre de risco (ao ano, em decimal). Ajuste se desejar (ex.: 0.065 para 6,5% a.a.)
RISK_FREE_ANUAL = 0.00

# ----------------------- CARGA & LIMPEZA -----------------------

def carregar_base_ibov(caminho, sheet=None):
    """
    Lê a planilha e retorna uma Série de preços do IBOV (^BVSP) com índice datetime.
    Identifica colunas de 'Date' e preço (prioriza '^BVSP'); trata datas seriais do Excel.
    """
    df = pd.read_excel(caminho, sheet_name=(0 if sheet is None else sheet), engine="openpyxl")
    df.columns = [str(c).strip() for c in df.columns]

    # Identifica coluna de data
    col_date = None
    for c in df.columns:
        if c.lower() in ("date", "data", "dt"):
            col_date = c
            break
    if col_date is None:
        raise ValueError("Coluna de data ('Date'/'Data'/'dt') não encontrada.")

    # Identifica coluna de preço (preferências)
    preferidas = ["^BVSP", "IBOV", "Close", "Adj Close", "Preço", "Price", "Fechamento"]
    col_price = next((c for c in preferidas if c in df.columns), None)
    if col_price is None:
        cand = [c for c in df.columns if c != col_date]
        if not cand:
            raise ValueError("Coluna de preço não encontrada.")
        col_price = cand[0]

    # Converte datas (suporta serial Excel e string)
    if np.issubdtype(df[col_date].dtype, np.number):
        df["Date"] = pd.to_datetime(df[col_date], unit="d", origin="1899-12-30", errors="coerce")
    else:
        df["Date"] = pd.to_datetime(df[col_date], errors="coerce")

    # Preço numérico; remove linhas ruins; ordena por data
    df["Price"] = pd.to_numeric(df[col_price], errors="coerce")
    df = df.dropna(subset=["Date", "Price"]).drop_duplicates(subset=["Date"]).sort_values("Date")

    # Série final
    s = df.set_index("Date")["Price"].astype(float).sort_index()
    return s

# ----------------------- FUNÇÕES AUXILIARES (Sharpe) -----------------------

def rf_diario(rf_aa: float) -> float:
    """Converte risco livre anual para equivalente diário (252 pregões)."""
    return (1 + rf_aa) ** (1/252) - 1

def rf_mensal(rf_aa: float) -> float:
    """Converte risco livre anual para equivalente mensal (12 meses)."""
    return (1 + rf_aa) ** (1/12) - 1

def sharpe_anualizado_diario(ret_d: pd.Series, rf_aa: float) -> float:
    """Sharpe anualizado com base diária (excess returns / std × sqrt(252))."""
    if len(ret_d) < 2:
        return np.nan
    rf_d = rf_diario(rf_aa)
    excess = ret_d - rf_d
    mu = excess.mean()
    sd = excess.std(ddof=1)
    if sd == 0 or np.isnan(sd):
        return np.nan
    return float((mu / sd) * np.sqrt(252))

def sharpe_anualizado_mensal(ret_m: pd.Series, rf_aa: float) -> float:
    """Sharpe anualizado com base mensal (excess returns / std × sqrt(12))."""
    if len(ret_m) < 2:
        return np.nan
    rf_m = rf_mensal(rf_aa)
    excess = ret_m - rf_m
    mu = excess.mean()
    sd = excess.std(ddof=1)
    if sd == 0 or np.isnan(sd):
        return np.nan
    return float((mu / sd) * np.sqrt(12))

# ----------------------- EXECUÇÃO -----------------------

if __name__ == "__main__":
    pd.set_option("display.max_rows", 200)
    pd.set_option("display.width", 160)

    # 1) Carrega série do IBOV (^BVSP)
    ibov = carregar_base_ibov(ARQ)

    # 2) Retornos
    ret_diario = ibov.pct_change().dropna()
    valor_mensal = ibov.resample("ME").last().dropna()   # último dia útil do mês
    ret_mensal = valor_mensal.pct_change().dropna()

    # 3) Sharpe do período completo
    sharpe_total_d = sharpe_anualizado_diario(ret_diario, RISK_FREE_ANUAL)
    sharpe_total_m = sharpe_anualizado_mensal(ret_mensal, RISK_FREE_ANUAL)

    # 4) Sharpe por ano
    sharpe_ano_d = []
    for ano, sub in ret_diario.groupby(ret_diario.index.year):
        s_ano = sharpe_anualizado_diario(sub, RISK_FREE_ANUAL)
        if not np.isnan(s_ano):
            sharpe_ano_d.append({"Ano": int(ano), "Sharpe (base diária)": s_ano})
    sharpe_ano_d = pd.DataFrame(sharpe_ano_d).sort_values("Ano")

    sharpe_ano_m = []
    for ano, sub in ret_mensal.groupby(ret_mensal.index.year):
        s_ano = sharpe_anualizado_mensal(sub, RISK_FREE_ANUAL)
        if not np.isnan(s_ano):
            sharpe_ano_m.append({"Ano": int(ano), "Sharpe (base mensal)": s_ano})
    sharpe_ano_m = pd.DataFrame(sharpe_ano_m).sort_values("Ano")

    # 5) Consolida por ano
    sharpe_anual = pd.merge(sharpe_ano_d, sharpe_ano_m, on="Ano", how="outer").sort_values("Ano")

    # 6) Impressões (Sharpe é adimensional; formatar com 4 casas)
    print("\n=== IBOV (^BVSP) — ÍNDICE DE SHARPE (ANUALIZADO) ===")
    print(f"Período na base: {ibov.index.min().date()} até {ibov.index.max().date()}  |  Nº pregões: {len(ibov)}")
    print(f"Taxa livre de risco (a.a.) considerada: {RISK_FREE_ANUAL*100:.2f}%")

    if not sharpe_anual.empty:
        sharpe_anual_fmt = sharpe_anual.copy()
        for c in ["Sharpe (base diária)", "Sharpe (base mensal)"]:
            if c in sharpe_anual_fmt.columns:
                sharpe_anual_fmt[c] = sharpe_anual_fmt[c].map(lambda x: f"{x:.4f}")
        print("\n--- Sharpe por ano ---")
        print(sharpe_anual_fmt.to_string(index=False))
    else:
        print("\n[Aviso] Não foi possível calcular Sharpe por ano (dados insuficientes).")

    print("\n--- Período completo ---")
    print(f"Sharpe (base diária): {sharpe_total_d:.4f}")
    print(f"Sharpe (base mensal): {sharpe_total_m:.4f}")


=== IBOV (^BVSP) — ÍNDICE DE SHARPE (ANUALIZADO) ===
Período na base: 2010-01-04 até 2019-12-30  |  Nº pregões: 2471
Taxa livre de risco (a.a.) considerada: 0.00%

--- Sharpe por ano ---
 Ano Sharpe (base diária) Sharpe (base mensal)
2010               0.0477               0.4180
2011              -0.6951              -1.0948
2012               0.4453               0.4478
2013              -0.7345              -1.0199
2014               0.0064              -0.0273
2015              -0.5120              -0.6038
2016               1.3820               1.3009
2017               1.3719               1.7502
2018               0.7593               0.7296
2019               1.6558               2.3148

--- Período completo ---
Sharpe (base diária): 0.3421
Sharpe (base mensal): 0.3908


# **# % MESES POSITIVOS - IBOVESPA**

In [6]:

# ----------------------- CARGA & LIMPEZA -----------------------

def carregar_base_ibov(caminho, sheet=None):
    """
    Lê a planilha e retorna uma Série de preços do IBOV (^BVSP) com índice datetime.
    Identifica colunas de 'Date' e preço (prioriza '^BVSP'); trata datas seriais do Excel.
    """
    df = pd.read_excel(caminho, sheet_name=(0 if sheet is None else sheet), engine="openpyxl")
    df.columns = [str(c).strip() for c in df.columns]

    # Identifica coluna de data
    col_date = None
    for c in df.columns:
        if c.lower() in ("date", "data", "dt"):
            col_date = c
            break
    if col_date is None:
        raise ValueError("Coluna de data ('Date'/'Data'/'dt') não encontrada.")

    # Identifica coluna de preço (preferências)
    preferidas = ["^BVSP", "IBOV", "Close", "Adj Close", "Preço", "Price", "Fechamento"]
    col_price = next((c for c in preferidas if c in df.columns), None)
    if col_price is None:
        cand = [c for c in df.columns if c != col_date]
        if not cand:
            raise ValueError("Coluna de preço não encontrada.")
        col_price = cand[0]

    # Converte datas (suporta serial Excel e string)
    if np.issubdtype(df[col_date].dtype, np.number):
        df["Date"] = pd.to_datetime(df[col_date], unit="d", origin="1899-12-30", errors="coerce")
    else:
        df["Date"] = pd.to_datetime(df[col_date], errors="coerce")

    # Preço numérico; remove linhas ruins; ordena por data
    df["Price"] = pd.to_numeric(df[col_price], errors="coerce")
    df = df.dropna(subset=["Date", "Price"]).drop_duplicates(subset=["Date"]).sort_values("Date")

    # Série final
    s = df.set_index("Date")["Price"].astype(float).sort_index()
    return s

# ----------------------- CÁLCULO % MESES POSITIVOS -----------------------

if __name__ == "__main__":
    pd.set_option("display.max_rows", 200)
    pd.set_option("display.width", 160)

    # 1) Carrega série do IBOV (^BVSP)
    ibov = carregar_base_ibov(ARQ)

    # 2) Retornos mensais pelo último dia útil do mês
    valor_mensal = ibov.resample("ME").last().dropna()
    ret_mensal = valor_mensal.pct_change().dropna()

    # 3) Percentual de meses positivos no período inteiro
    total_meses = len(ret_mensal)
    meses_positivos = int((ret_mensal > 0).sum())
    perc_positivos_total = (meses_positivos / total_meses * 100.0) if total_meses > 0 else float("nan")

    print("\n=== IBOV (^BVSP) — MESES POSITIVOS ===")
    print(f"Período na base: {ibov.index.min().date()} até {ibov.index.max().date()}  |  Nº pregões: {len(ibov)}")
    print(f"\nMeses positivos (período): {meses_positivos} de {total_meses}  ->  {perc_positivos_total:.2f}%")

    # 4) (Opcional) Percentual de meses positivos por ano
    positivos_por_ano = []
    for ano, sub in ret_mensal.groupby(ret_mensal.index.year):
        n = len(sub)
        k = int((sub > 0).sum())
        perc = (k / n * 100.0) if n > 0 else float("nan")
        positivos_por_ano.append({"Ano": int(ano), "Meses Positivos": k, "Total de Meses": n, "% Meses Positivos": perc})

    if positivos_por_ano:
        df_pos_ano = pd.DataFrame(positivos_por_ano).sort_values("Ano")
        # Formata a coluna percentual com 2 casas
        df_pos_ano["% Meses Positivos"] = df_pos_ano["% Meses Positivos"].map(lambda x: f"{x:.2f}%")
        print("\n--- Por ano ---")
        print(df_pos_ano.to_string(index=False))


=== IBOV (^BVSP) — MESES POSITIVOS ===
Período na base: 2010-01-04 até 2019-12-30  |  Nº pregões: 2471

Meses positivos (período): 64 de 119  ->  53.78%

--- Por ano ---
 Ano  Meses Positivos  Total de Meses % Meses Positivos
2010                6              11            54.55%
2011                3              12            25.00%
2012                7              12            58.33%
2013                4              12            33.33%
2014                7              12            58.33%
2015                4              12            33.33%
2016                8              12            66.67%
2017                9              12            75.00%
2018                7              12            58.33%
2019                9              12            75.00%


# **# EXCEL - RETORNO MENSAL IBOVESPA**

In [7]:
#RET_MES_IBOV = ret_mensal_df


#Ret_Mes_Ibov = ret_mensal_df.iloc[:, 2].astype(float).mul(100).map(lambda x: f"{x:.2f}%")

#RET_MES_IBOV.to_excel('9_1_RET_MES_IBOV.xlsx', index=False, engine='openpyxl')   # Criando arquivo Excel ReesultadoFM.xlxl
#print("Arquivo Excel 9_1_RET_MES_IBOV criado com sucesso!")    

In [8]:

#RET_MES_IBOV = ret_mensal_df.copy()
#RET_MES_IBOV["Retorno Mensal (%)"] = RET_MES_IBOV["Retorno Mensal"].astype(float).map(lambda x: f"{x:.2%}")
#RET_MES_IBOV = RET_MES_IBOV.drop(columns=["Retorno Mensal"])



In [9]:


RET_MES_IBOV = ret_mensal_df.copy()

arquivo_saida = '0_8_1_RET_MES_IBOV.xlsx'
nome_aba = 'RET_MES_IBOV'
coluna_percentual = 'Retorno Mensal' 
with pd.ExcelWriter(arquivo_saida, engine='openpyxl') as writer:
    RET_MES_IBOV.to_excel(writer, index=False, sheet_name=nome_aba)

    ws = writer.sheets[nome_aba]
    

    # *Descobre o índice (1-based) da coluna "Retorno Mensal"

    try:
        col_idx = RET_MES_IBOV.columns.get_loc(coluna_percentual) + 1
    except KeyError:
        raise KeyError(f'A coluna "{coluna_percentual}" não foi encontrada no DataFrame. '
                       f'Colunas existentes: {list(RET_MES_IBOV.columns)}')


    # *Aplica formatação de porcentagem (duas casas) da linha 2 até a última (linha 1 é o cabeçalho)

    for col in ws.iter_cols(min_col=col_idx, max_col=col_idx, min_row=2, max_row=ws.max_row):
        for cell in col:
            cell.number_format = '0.00%'

print(f"Arquivo Excel {arquivo_saida} criado com sucesso!")


Arquivo Excel 0_8_1_RET_MES_IBOV.xlsx criado com sucesso!


In [10]:
Arq_RET_MES_IBOV = op.load_workbook('0_8_1_RET_MES_IBOV.xlsx')                                 # *Carregando arquivo Excel 9_1_RET_MES_IBOV.xlxl
Plan_RET_MES_IBOV = Arq_RET_MES_IBOV['RET_MES_IBOV']                                     # *Carregando planilha Excel expec[ifica em Ree9_1_RET_MES_IBOVsultadoFM.xlxl
Arq_RET_MES_IBOV

In [11]:
Arqler_RET_MES_IBOV = pd.read_excel('0_8_1_RET_MES_IBOV.xlsx')                                 # *Lendo arquivo Excel 9_1_RET_MES_IBOV.xlxl
Arqler_RET_MES_IBOV

,Ano,Mes,Retorno Mensal
0,2010,2,0.016834
1,2010,3,0.058178
2,2010,4,-0.040385
3,2010,5,-0.066385
4,2010,6,-0.033483
5,2010,7,0.107966
6,2010,8,-0.035103
7,2010,9,0.065776
8,2010,10,0.017903
9,2010,11,-0.041996


# **# EXCEL - RETORNO ANUAL IBOVESPA**

In [12]:


RET_ANO_IBOV = ret_anual_df.copy()

arquivo_saida_ano = '0_8_2_RET_ANO_IBOV.xlsx'
nome_aba = 'RET_ANO_IBOV'
coluna_percentual_ano = 'Retorno Anual (Jan→Dez)'  \

with pd.ExcelWriter(arquivo_saida_ano, engine='openpyxl') as writer:
    RET_ANO_IBOV.to_excel(writer, index=False, sheet_name=nome_aba)

    ws = writer.sheets[nome_aba]


 # *Descobre o índice (1-based) da coluna "Retorno ANUAL"

    try:
        col_idx = RET_ANO_IBOV.columns.get_loc(coluna_percentual_ano) + 1
    except KeyError:
        raise KeyError(f'A coluna "{coluna_percentual_ano}" não foi encontrada no DataFrame. '
                       f'Colunas existentes: {list(RET_ANO_IBOV.columns)}')


# *Aplica formatação de porcentagem (duas casas) da linha 2 até a última (linha 1 é o cabeçalho)

    for col in ws.iter_cols(min_col=col_idx, max_col=col_idx, min_row=2, max_row=ws.max_row):
        for cell in col:
            cell.number_format = '0.00%'

print(f"Arquivo Excel {arquivo_saida_ano} criado com sucesso!")


Arquivo Excel 0_8_2_RET_ANO_IBOV.xlsx criado com sucesso!


In [13]:
Arq_RET_ANO_IBOV = op.load_workbook('0_8_2_RET_ANO_IBOV.xlsx')                                 # *Carregando arquivo Excel 9_2_RET_ANO_IBOV.xlxl
Plan_RET_ANO_IBOV = Arq_RET_ANO_IBOV['RET_ANO_IBOV']                                     # *Carregando planilha Excel expec[ifica em Ree9_2_RET_ANO_IBOV.xlxl
Arq_RET_ANO_IBOV

In [14]:
Arqler_RET_ANO_IBOV = pd.read_excel('0_8_2_RET_ANO_IBOV.xlsx')                # *Lendo arquivo Excel 9_2_RET_ANO_IBOV.xlxl
Arqler_RET_ANO_IBOV

,Ano,Retorno Anual (Jan→Dez)
0,2010,-0.010565
1,2011,-0.188788
2,2012,0.028465
3,2013,-0.176547
4,2014,-0.006635
5,2015,-0.106407
6,2016,0.429178
7,2017,0.282149
8,2018,0.128333
9,2019,0.274162


# **# EXCEL - VOLATILIDADE ANUAL IBOVESPA - MÉDIA DIÁRIA/MENSAL**

In [15]:
VOL_ANO_IBOV = vol_anual_fmt

VOL_ANO_IBOV.to_excel('0_8_3_VOL_ANO_IBOV.xlsx', index=False, engine='openpyxl')         # *Criando arquivo Excel 9_3_VOL_ANO_IBOV.xlsx
print("Arquivo Excel 0_8_3_VOL_ANO_IBOV criado com sucesso!")                            # *Confirmando a criação do arquivo excel 9_3_VOL_ANO_IBOV.xlsx

Arquivo Excel 0_8_3_VOL_ANO_IBOV criado com sucesso!


In [16]:
Arq_VOL_ANO_IBOV = op.load_workbook('0_8_3_VOL_ANO_IBOV.xlsx')                           # *Carregando arquivo Excel 9_3_VOL_ANO_IBOV.xlsx
Arq_VOL_ANO_IBOV = Arq_VOL_ANO_IBOV['Sheet1']                                          # *Carregando planilha Excel expec[ifica em 9_3_VOL_ANO_IBOV.xlsx
Arq_VOL_ANO_IBOV

<Worksheet "Sheet1">

In [17]:
ArqLer_Arq_VOL_ANO_D_IBOV = pd.read_excel('0_8_3_VOL_ANO_IBOV.xlsx')                  # *Lendo arquivo Excel 9_3_VOL_ANO_IBOV.xlsx
ArqLer_Arq_VOL_ANO_D_IBOV

,Ano,Vol Anualizada (base diária),Vol Anualizada (base mensal)
0,2010,20.30%,19.04%
1,2011,24.68%,16.93%
2,2012,21.89%,20.22%
3,2013,20.45%,15.30%
4,2014,25.22%,22.38%
5,2015,23.30%,20.42%
6,2016,26.65%,28.39%
7,2017,19.03%,14.25%
8,2018,22.21%,22.45%
9,2019,18.00%,12.41%


# **# EXCEL - DRAWDOWN MÁXIMO IBOVESPA**

In [18]:
MDD_IBOV = mdd_por_ano

MDD_IBOV.to_excel('0_8_4_MDD_IBOV.xlsx', index=False, engine='openpyxl')               # *Criando arquivo Excel 9_4_MDD_IBOV.xlsx
print("Arquivo Excel 0_8_4_MDD_IBOV criado com sucesso!")                              # *Confirmando a criação do arquivo excel 9_4_MDD_IBOV.xlsx

Arquivo Excel 0_8_4_MDD_IBOV criado com sucesso!


In [19]:
Arq_MDD_IBOV = op.load_workbook('0_8_4_MDD_IBOV.xlsx')                                # *Carregando arquivo Excel 9_4_MDD_IBOV.xlsx
Arq_MDD_IBOV = Arq_MDD_IBOV['Sheet1']                                               # *Carregando planilha Excel expec[ifica em 9_4_MDD_IBOV.xlsx
Arq_MDD_IBOV

<Worksheet "Sheet1">

In [20]:
ArqLer_MDD_IBOV = pd.read_excel('0_8_4_MDD_IBOV.xlsx')                                # *Lendo arquivo Excel 9_4_MDD_IBOV.xlsx
ArqLer_MDD_IBOV

,Ano,MDD,Peak,Trough
0,2010,-0.189357,2010-04-08,2010-05-20
1,2011,-0.320592,2011-01-12,2011-08-08
2,2012,-0.232667,2012-03-13,2012-06-05
3,2013,-0.288539,2013-01-03,2013-07-03
4,2014,-0.240533,2014-09-02,2014-12-16
5,2015,-0.255840,2015-05-05,2015-12-21
6,2016,-0.120354,2016-10-31,2016-12-19
7,2017,-0.120054,2017-02-21,2017-06-21
8,2018,-0.203507,2018-02-26,2018-06-18
9,2019,-0.100016,2019-03-18,2019-05-17


# **# EXCEL - ÍNDICE SHARPE IBOVESPA**

In [21]:
SHARPE_IBOV = sharpe_anual
SHARPE_IBOV.to_excel('0_8_5_SHARPE_IBOV.xlsx', index=False, engine='openpyxl')               # *Criando arquivo Excel 0_8_5_SHARPE_IBOV.xlsx
print("Arquivo Excel 0_8_5_SHARPE_IBOV criado com sucesso!")                                 # *Confirmando a criação do arquivo excel 0_8_5_SHARPE_IBOV.xlsx

Arquivo Excel 0_8_5_SHARPE_IBOV criado com sucesso!


In [22]:
Arq_SHARPE_IBOV = op.load_workbook('0_8_5_SHARPE_IBOV.xlsx')                                # *Carregando arquivo Excel 0_8_5_SHARPE_IBOV.xlsx
Arq_SHARPE_IBOV = Arq_SHARPE_IBOV['Sheet1']                                               # *Carregando planilha Excel expec[ifica em 0_8_5_SHARPE_IBOV.xlsx
Arq_SHARPE_IBOV

<Worksheet "Sheet1">

In [23]:
ArqLer_SHARPE_IBOV = pd.read_excel('0_8_5_SHARPE_IBOV.xlsx')                                # *Lendo arquivo Excel 0_8_5_SHARPE_IBOV.xlsx
ArqLer_SHARPE_IBOV

,Ano,Sharpe (base diária),Sharpe (base mensal)
0,2010,0.047705,0.417950
1,2011,-0.695094,-1.094779
2,2012,0.445346,0.447825
3,2013,-0.734496,-1.019947
4,2014,0.006351,-0.027311
5,2015,-0.512027,-0.603751
6,2016,1.381957,1.300941
7,2017,1.371916,1.750200
8,2018,0.759258,0.729637
9,2019,1.655762,2.314818


# **# EXCEL - % MESES POSITIVOS IBOVESPA**

In [27]:
POSITIVOS_ANO_IBOV = df_pos_ano

POSITIVOS_ANO_IBOV = POSITIVOS_ANO_IBOV.copy()
POSITIVOS_ANO_IBOV.to_excel('0_8_6_POSITIVOS_ANO_IBOV.xlsx', index=False, engine='openpyxl')               # *Criando arquivo Excel 0_8_6_POSITIVOS_ANO_IBOV.xlsx
print("Arquivo Excel 0_8_6_POSITIVOS_ANO_IBOV criado com sucesso!")  

Arquivo Excel 0_8_6_POSITIVOS_ANO_IBOV criado com sucesso!


In [28]:
Arq_POSITIVOS_ANO_IBOV = op.load_workbook('0_8_6_POSITIVOS_ANO_IBOV.xlsx')                                # *Carregando arquivo Excel 0_8_6_POSITIVOS_ANO_IBOV.xlsx
Arq_POSITIVOS_ANO_IBOV = Arq_POSITIVOS_ANO_IBOV['Sheet1']                                               # *Carregando planilha Excel expec[ifica em 0_8_6_POSITIVOS_ANO_IBOV.xlsx
Arq_POSITIVOS_ANO_IBOV

<Worksheet "Sheet1">

In [29]:
ArqLer_POSITIVOS_ANO_IBOV = pd.read_excel('0_8_6_POSITIVOS_ANO_IBOV.xlsx')                                # *Lendo arquivo Excel 0_8_6_POSITIVOS_ANO_IBOV.xlsx
ArqLer_POSITIVOS_ANO_IBOV

,Ano,Meses Positivos,Total de Meses,% Meses Positivos
0,2010,6,11,54.55%
1,2011,3,12,25.00%
2,2012,7,12,58.33%
3,2013,4,12,33.33%
4,2014,7,12,58.33%
5,2015,4,12,33.33%
6,2016,8,12,66.67%
7,2017,9,12,75.00%
8,2018,7,12,58.33%
9,2019,9,12,75.00%
